In [ ]:
pip install langchain-ollama

In [ ]:
pip install grandalf

In [1]:
from langchain_ollama import ChatOllama
model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)

# Simple Chain

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    template= "Generate 3 intersting fact about {person}",
    input_variables=["person"]
)

parser = StrOutputParser()

chain = prompt | model | parser
result = chain.invoke({"person": "APJ Abdul Kalam"})
print(result)
chain.get_graph().print_ascii()

Here are three interesting facts about **APJ Abdul Kalam**:

1. **First Indian President with a PhD**: APJ Abdul Kalam was the first Indian president to hold a PhD degree in engineering. He earned his doctorate in aeronautics from the Indian Institute of Technology, Kharagpur, in 1963, making him a pioneer in India's scientific and educational landscape.

2. **Pilot and Scientist**: Before becoming India's president, Kalam was a test pilot and engineer. He famously flew the **Agni** missile, India's first intercontinental ballistic missile, during his military career, showcasing his expertise in aerospace technology.

3. **Cricket Enthusiast**: Despite his monumental achievements, Kalam was a passionate cricket fan. He often played cricket in his later years, and his love for the sport was a testament to his humble, patriotic spirit. He even wrote a book titled *The Cricket Cricketer* about his experiences playing the game. 

These facts highlight his dual legacy as a visionary leader 

### Sequencial Chain

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Slogans(BaseModel):
    slogans: list[str] = Field(description="List of marketing slogans", min_length=5, max_length=5)

parser = PydanticOutputParser(pydantic_object=Slogans)

prompt1 = PromptTemplate(
    template= "Generate a creative product description for a product : {product_name}",
    input_variables=["product_name"])

prompt2 = PromptTemplate(
    template= "Generate 5 unique and catchy marketing slogans for a product description: \n {product_description}\n{format_instructions}",
    input_variables=["product_description"],
    partial_variables={"format_instructions": parser.get_format_instructions()})

chain = prompt1 | model | prompt2 | model | parser
result = chain.invoke({"product_name": "Samsung Galaxy S23 Ultra Smartphone with S-Pen"})
for slogan in result.slogans:
    print("-", slogan)
chain.get_graph().print_ascii()

- S-Pen 2.0: Your Digital Canvas for Creativity
- Immerse Yourself in Infinite Detail with 120Hz AMOLED
- Unleash Power, Creativity, and Precision with the S23 Ultra
- 5G Speed, AI Intelligence – Transform Your Work
- Elevate Your Visuals with 120Hz AMOLED Display
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
       

### Parallel Chaining 

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableParallel

model1 = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)

model2 = ChatOllama(
    model="qwen3-vl:8b",
    temperature=0,
)
prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)
prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)
prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)
parser = StrOutputParser()
parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

ModuleNotFoundError: No module named 'langchain'